# ARI711S – Artificial Intelligence — Assignment 1
**Faculty of Computing and Informatics | Department of Software Engineering**  
**Qualification:** Bachelor of Computer Science (Software Development)  
**Due Date:** 26 April 2026 @ 23:59

---
This notebook answers all four questions using Python, with explanations via Markdown and full visual outputs.


## Question 1: Search Algorithms – Warehouse Logistics Bot

### Background
We implement **Greedy Best-First Search** and **A\* Search** to navigate an Automated Guided Vehicle (AGV) from a Charging Station **A** to a Product Bin **B** inside a warehouse grid.

### Problem Formulation
| Component | Description |
|-----------|-------------|
| **States** | Grid coordinates `(row, col)` |
| **Actions** | `up, down, left, right` (non-wall cells only) |
| **Initial state** | Cell marked `A` |
| **Goal state** | Cell marked `B` |
| **Heuristic** | Euclidean distance: h(n) = √((x₁−x₂)² + (y₁−y₂)²) |

### Algorithm Comparison
| Algorithm | Priority | Guarantee |
|-----------|----------|-----------|
| Greedy | h(n) only | Fast, not always optimal |
| A\* | f(n)=g(n)+h(n) | Optimal (with admissible h) |


In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

# ──────────────────────────────────────────────
# Node class
# ──────────────────────────────────────────────
class Node:
    def __init__(self, state, parent=None, action=None, g=0):
        self.state  = state   # (row, col)
        self.parent = parent
        self.action = action  # 'up','down','left','right'
        self.g      = g       # path cost so far

    def __lt__(self, other):
        return False          # tie-break for heapq


# ──────────────────────────────────────────────
# Warehouse class
# ──────────────────────────────────────────────
class Warehouse:
    def __init__(self, filepath):
        self.walls  = set()
        self.start  = None
        self.goal   = None
        self.grid   = []
        self._parse(filepath)

    def _parse(self, filepath):
        # If the file doesn't exist, create a sample warehouse for demo
        if not os.path.exists(filepath):
            print(f"[INFO] {filepath} not found – using built-in sample warehouse.")
            layout = [
                "####################",
                "#A   #    #    #   B#",
                "#    #    #    #    #",
                "#         #         #",
                "#    ###########    #",
                "#         #         #",
                "#    #    #    #    #",
                "#    #         #    #",
                "####################",
            ]
        else:
            with open(filepath) as f:
                layout = [line.rstrip('\n') for line in f]

        self.grid = layout
        for r, row in enumerate(layout):
            for c, ch in enumerate(row):
                if ch == '#':
                    self.walls.add((r, c))
                elif ch == 'A':
                    self.start = (r, c)
                elif ch == 'B':
                    self.goal  = (r, c)

        self.rows = len(layout)
        self.cols = max(len(row) for row in layout)
        assert self.start, "No 'A' found in warehouse layout"
        assert self.goal,  "No 'B' found in warehouse layout"

    # ── heuristic ──────────────────────────────
    def heuristic(self, state):
        x1, y1 = state
        x2, y2 = self.goal
        return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    # ── neighbors ──────────────────────────────
    def neighbors(self, state):
        r, c = state
        result = []
        for action, (dr, dc) in [('up',(-1,0)),('down',(1,0)),
                                   ('left',(0,-1)),('right',(0,1))]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols \
               and (nr, nc) not in self.walls:
                result.append((action, (nr, nc)))
        return result

    # ── solve ──────────────────────────────────
    def solve(self, algorithm="astar"):
        start_node = Node(self.start, g=0)
        h0 = self.heuristic(self.start)
        priority = h0 if algorithm == "greedy" else 0 + h0

        frontier  = []
        heapq.heappush(frontier, (priority, start_node))

        explored   = set()
        frontier_states = {self.start}

        while frontier:
            _, node = heapq.heappop(frontier)
            frontier_states.discard(node.state)

            if node.state == self.goal:
                return self._extract_path(node), explored

            if node.state in explored:
                continue
            explored.add(node.state)

            for action, next_state in self.neighbors(node.state):
                if next_state not in explored and next_state not in frontier_states:
                    g_new = node.g + 1
                    h_new = self.heuristic(next_state)
                    p_new = h_new if algorithm == "greedy" else g_new + h_new
                    child = Node(next_state, parent=node, action=action, g=g_new)
                    heapq.heappush(frontier, (p_new, child))
                    frontier_states.add(next_state)

        return None, explored   # no path found

    # ── path extraction ────────────────────────
    def _extract_path(self, node):
        path = []
        while node.parent:
            path.append(node.state)
            node = node.parent
        path.append(self.start)
        path.reverse()
        return path

    # ── visualise ──────────────────────────────
    def visualise(self, path, explored, filename="warehouse_path.png", title=""):
        path_set = set(path)
        img = np.ones((self.rows, self.cols, 3))

        color_map = {
            'wall':     [0.2, 0.2, 0.2],   # dark gray
            'explored': [0.8, 0.9, 1.0],   # light blue
            'path':     [0.0, 0.8, 0.0],   # green
            'start':    [1.0, 0.6, 0.0],   # orange
            'goal':     [1.0, 0.1, 0.1],   # red
            'open':     [1.0, 1.0, 1.0],   # white
        }

        for r in range(self.rows):
            for c in range(self.cols):
                state = (r, c)
                if state in self.walls:
                    img[r, c] = color_map['wall']
                elif state == self.start:
                    img[r, c] = color_map['start']
                elif state == self.goal:
                    img[r, c] = color_map['goal']
                elif state in path_set:
                    img[r, c] = color_map['path']
                elif state in explored:
                    img[r, c] = color_map['explored']

        fig, ax = plt.subplots(figsize=(max(8, self.cols * 0.5),
                                        max(6, self.rows * 0.5)))
        ax.imshow(img, interpolation='nearest')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

        legend = [
            mpatches.Patch(color=color_map['wall'],     label='Shelving (Wall)'),
            mpatches.Patch(color=color_map['path'],     label='Optimal Path'),
            mpatches.Patch(color=color_map['explored'], label='Explored'),
            mpatches.Patch(color=color_map['start'],    label='Charging Station (A)'),
            mpatches.Patch(color=color_map['goal'],     label='Product Bin (B)'),
        ]
        ax.legend(handles=legend, loc='upper right', fontsize=8,
                  bbox_to_anchor=(1.25, 1.0))
        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"[Saved] {filename}")


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
if __name__ == "__main__":
    wh = Warehouse("warehouse.txt")

    for algo in ["greedy", "astar"]:
        path, explored = wh.solve(algorithm=algo)
        if path:
            print(f"\n[{algo.upper()}]")
            print(f"  Path length : {len(path)} cells")
            print(f"  Explored    : {len(explored)} states")
            print(f"  Path        : {path}")
            fname = f"warehouse_path_{algo}.png"
            wh.visualise(path, explored, filename=fname,
                         title=f"Warehouse – {algo.upper()} Search\n"
                               f"Path={len(path)} cells  |  Explored={len(explored)} states")
        else:
            print(f"[{algo.upper()}] No path found.")



In [ ]:
# Run both algorithms
wh = Warehouse('warehouse.txt')
for algo in ['greedy','astar']:
    path, explored = wh.solve(algorithm=algo)
    print(f'[{algo.upper()}]  Path={len(path)} cells  Explored={len(explored)} states')
    fname = f'warehouse_path_{algo}.png'
    wh.visualise(path, explored, filename=fname,
                 title=f'Warehouse – {algo.upper()} Search\nPath={len(path)} cells | Explored={len(explored)} states')


### Results
The images `warehouse_path_greedy.png` and `warehouse_path_astar.png` show:
- 🟢 **Green** – optimal path
- 🔵 **Light blue** – explored (visited) states  
- ⬛ **Dark grey** – shelving / walls  
- 🟠 **Orange** – Charging Station A  
- 🔴 **Red** – Product Bin B

**Key insight:** A\* explores more nodes than Greedy in some cases, but guarantees the *shortest* path, while Greedy may find a suboptimal route quickly.


---
## Question 2: Optimisation – Telecommunication Tower Placement (25 marks)

### Background
MTC must place **8 signal boosters** on a 10×10 grid without violating spatial constraints. This is a **Constraint Satisfaction Problem (CSP)**.

### Constraints
1. **No shared row or column** (signal overlap)
2. **No adjacent cells** including diagonals (interference buffer ≥ 2)
3. **No mountain cells** (terrain obstruction)

### Implementation Highlights
- **`backtrack(assignment)`** – recursive DFS with pruning
- **MRV heuristic** – pick the variable with fewest remaining values
- **Forward Checking** – after each placement, prune other domains


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import copy

# ──────────────────────────────────────────────
# Telecom CSP Solver
# ──────────────────────────────────────────────
class Telecom_CSP_Solver:
    def __init__(self, mountains=None, grid_size=10, num_towers=8):
        self.grid_size  = grid_size
        self.num_towers = num_towers
        self.mountains  = set(mountains) if mountains else set()
        self.towers     = [f"T{i+1}" for i in range(num_towers)]

        # Initial domain: all non-mountain cells
        all_cells = [(r, c) for r in range(grid_size)
                             for c in range(grid_size)
                             if (r, c) not in self.mountains]
        self.initial_domains = {t: list(all_cells) for t in self.towers}

    # ── constraint check ──────────────────────
    def is_consistent(self, assignment, cell):
        """Return True if placing a new tower at 'cell' is consistent with existing assignment."""
        r, c = cell
        for placed_cell in assignment.values():
            pr, pc = placed_cell
            # same row or column
            if pr == r or pc == c:
                return False
            # adjacency (including diagonal) within 1 step
            if abs(pr - r) <= 1 and abs(pc - c) <= 1:
                return False
        return True

    # ── forward checking ──────────────────────
    def forward_check(self, domains, tower, cell):
        """Prune domains of unassigned towers after placing tower at cell."""
        new_domains = copy.deepcopy(domains)
        r, c = cell
        for other in self.towers:
            if other in new_domains:   # unassigned
                pruned = []
                for (pr, pc) in new_domains[other]:
                    if pr == r or pc == c:
                        continue
                    if abs(pr - r) <= 1 and abs(pc - c) <= 1:
                        continue
                    pruned.append((pr, pc))
                new_domains[other] = pruned
                if not pruned:
                    return None   # domain wipe-out
        return new_domains

    # ── MRV heuristic ─────────────────────────
    def select_unassigned(self, assignment, domains):
        """Choose the tower with the fewest remaining valid cells (MRV)."""
        unassigned = [t for t in self.towers if t not in assignment]
        return min(unassigned, key=lambda t: len(domains[t]))

    # ── backtracking search ───────────────────
    def backtrack(self, assignment, domains):
        if len(assignment) == self.num_towers:
            return assignment

        tower = self.select_unassigned(assignment, domains)

        for cell in list(domains[tower]):
            if self.is_consistent(assignment, cell):
                assignment[tower] = cell

                # remove tower from domains (it's now assigned)
                new_domains = {t: v for t, v in domains.items() if t != tower}
                new_domains = self.forward_check(new_domains, tower, cell)

                if new_domains is not None:
                    result = self.backtrack(assignment, new_domains)
                    if result is not None:
                        return result

                del assignment[tower]

        return None

    # ── solve ─────────────────────────────────
    def solve(self):
        solution = self.backtrack({}, copy.deepcopy(self.initial_domains))
        return solution

    # ── visualise ─────────────────────────────
    def visualise(self, solution, title="MTC 5G Tower Placement", filename="towers.png"):
        fig, ax = plt.subplots(figsize=(8, 8))
        gs = self.grid_size

        # background
        ax.set_facecolor('#f5f5f5')

        # grid lines
        for i in range(gs + 1):
            ax.axhline(i, color='gray', linewidth=0.5)
            ax.axvline(i, color='gray', linewidth=0.5)

        # mountain cells
        for (r, c) in self.mountains:
            ax.add_patch(plt.Rectangle((c, gs - 1 - r), 1, 1,
                                        color='#8B4513', zorder=2))
            ax.text(c + 0.5, gs - 1 - r + 0.5, 'M',
                    ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white', zorder=3)

        # towers
        if solution:
            for name, (r, c) in solution.items():
                ax.add_patch(plt.Rectangle((c, gs - 1 - r), 1, 1,
                                            color='#1565C0', zorder=2))
                ax.text(c + 0.5, gs - 1 - r + 0.5, 'T',
                        ha='center', va='center',
                        fontsize=11, fontweight='bold', color='white', zorder=3)

        ax.set_xlim(0, gs)
        ax.set_ylim(0, gs)
        ax.set_xticks(range(gs))
        ax.set_yticks(range(gs))
        ax.set_xticklabels(range(gs))
        ax.set_yticklabels(range(gs - 1, -1, -1))
        ax.set_title(title, fontsize=13, fontweight='bold')

        legend = [
            mpatches.Patch(color='#1565C0', label='Tower (T)'),
            mpatches.Patch(color='#8B4513', label='Mountain (M)'),
        ]
        ax.legend(handles=legend, loc='upper right', fontsize=9)
        plt.tight_layout()
        plt.savefig(filename, dpi=150)
        plt.close()
        print(f"[Saved] {filename}")


# ──────────────────────────────────────────────
# Test Scenarios
# ──────────────────────────────────────────────
scenarios = {
    "Level1_Coastal": [(0,0),(1,1),(9,9)],
    "Level2_Highlands": [(2,2),(2,3),(3,2),(3,3),(7,8),(8,7),(8,8)],
    "Level3_Brandberg": [(0,5),(1,5),(2,5),(3,5),(4,5),
                          (5,0),(5,1),(5,2),(5,3),(5,4)],
}

if __name__ == "__main__":
    for name, mountains in scenarios.items():
        print(f"\n{'='*50}")
        print(f"Solving: {name}")
        solver = Telecom_CSP_Solver(mountains=mountains)
        solution = solver.solve()

        if solution:
            print(f"  Solution found:")
            for t, cell in sorted(solution.items()):
                print(f"    {t} → {cell}")
            solver.visualise(solution,
                             title=f"MTC Tower Placement – {name.replace('_',' ')}",
                             filename=f"towers_{name}.png")
        else:
            print("  No solution found.")



In [ ]:
scenarios = {
    'Level1_Coastal':   [(0,0),(1,1),(9,9)],
    'Level2_Highlands': [(2,2),(2,3),(3,2),(3,3),(7,8),(8,7),(8,8)],
    'Level3_Brandberg': [(0,5),(1,5),(2,5),(3,5),(4,5),(5,0),(5,1),(5,2),(5,3),(5,4)],
}
for name, mountains in scenarios.items():
    print(f'--- {name} ---')
    solver = Telecom_CSP_Solver(mountains=mountains)
    sol = solver.solve()
    if sol:
        for t,cell in sorted(sol.items()): print(f'  {t} -> {cell}')
        solver.visualise(sol, title=f'MTC Towers – {name}', filename=f'towers_{name}.png')
    else:
        print('  No solution')


---
## Question 3: Adversarial Search – Tic-Tac-Toe with Minimax

### Background
We build an AI agent that plays Tic-Tac-Toe **optimally** using the **Minimax algorithm** with **alpha-beta pruning** for efficiency.

### Game Formulation
| Component | Description |
|-----------|-------------|
| States | 3×3 boards |
| Actions | Empty cells (i, j) |
| Players | X (maximiser), O (minimiser) |
| Terminal | Full board or winner |
| Utility | X wins: +1 | O wins: −1 | Tie: 0 |

### Files
- **`tictactoe.py`** – all game logic + Minimax
- **`runner.py`** – Tkinter GUI (run with `python runner.py`)


In [ ]:
"""
tictactoe.py  –  Minimax Tic-Tac-Toe AI
"""
import copy
import math

X     = "X"
O     = "O"
EMPTY = None

# ──────────────────────────────────────────────
def initial_state():
    """Returns the starting empty 3×3 board."""
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

# ──────────────────────────────────────────────
def player(board):
    """Return whose turn it is (X or O)."""
    x_count = sum(row.count(X) for row in board)
    o_count = sum(row.count(O) for row in board)
    return X if x_count == o_count else O

# ──────────────────────────────────────────────
def actions(board):
    """Return set of all valid (i, j) actions."""
    return {(i, j)
            for i in range(3)
            for j in range(3)
            if board[i][j] == EMPTY}

# ──────────────────────────────────────────────
def result(board, action):
    """Return new board after applying action (deep copy, no mutation)."""
    i, j = action
    if board[i][j] != EMPTY:
        raise ValueError(f"Invalid action {action}: cell is already occupied.")
    new_board = copy.deepcopy(board)
    new_board[i][j] = player(board)
    return new_board

# ──────────────────────────────────────────────
def winner(board):
    """Return X, O, or None."""
    lines = (
        # rows
        [board[0], board[1], board[2]],
        # cols
        [[board[r][c] for r in range(3)] for c in range(3)],
        # diagonals
        [[board[i][i] for i in range(3)]],
        [[board[i][2-i] for i in range(3)]],
    )
    for group in lines:
        for line in group:
            if line[0] is not None and line[0] == line[1] == line[2]:
                return line[0]
    return None

# ──────────────────────────────────────────────
def terminal(board):
    """Return True if the game is over."""
    if winner(board) is not None:
        return True
    return all(board[i][j] != EMPTY for i in range(3) for j in range(3))

# ──────────────────────────────────────────────
def utility(board):
    """Return +1 (X wins), -1 (O wins), or 0 (tie). Only call on terminal boards."""
    w = winner(board)
    if w == X:
        return 1
    elif w == O:
        return -1
    return 0

# ──────────────────────────────────────────────
# Minimax with optional alpha-beta pruning
# ──────────────────────────────────────────────
def _max_value(board, alpha, beta):
    if terminal(board):
        return utility(board), None
    v    = -math.inf
    best = None
    for action in actions(board):
        val, _ = _min_value(result(board, action), alpha, beta)
        if val > v:
            v, best = val, action
        alpha = max(alpha, v)
        if alpha >= beta:
            break           # β cut-off
    return v, best

def _min_value(board, alpha, beta):
    if terminal(board):
        return utility(board), None
    v    = math.inf
    best = None
    for action in actions(board):
        val, _ = _max_value(result(board, action), alpha, beta)
        if val < v:
            v, best = val, action
        beta = min(beta, v)
        if alpha >= beta:
            break           # α cut-off
    return v, best

def minimax(board):
    """Return the optimal action for the current player, or None if terminal."""
    if terminal(board):
        return None
    if player(board) == X:
        _, action = _max_value(board, -math.inf, math.inf)
    else:
        _, action = _min_value(board, -math.inf, math.inf)
    return action



In [ ]:
# Demo: simulate a full game AI vs AI
board = initial_state()
move_num = 0
while not terminal(board):
    move = minimax(board)
    board = result(board, move)
    move_num += 1
    print(f'Move {move_num} by {"X" if move_num%2==1 else "O"} at {move}:')
    for row in board: print(' '.join(c if c else '.' for c in row))
    print()
w = winner(board)
print(f'Result: {w} wins!' if w else 'Result: Tie!')


### Runner
Run the GUI with:
```bash
python runner.py
```
The human plays as **O**, the AI plays as **X** (goes first).  
Alpha-beta pruning ensures moves are computed instantly.


---
## Question 4: MDPs – SARSA (25 marks)

### Background
We implement the **SARSA (on-policy TD)** algorithm on the classic 5×5 Gridworld MDP.

| State | Reward | Transition |
|-------|--------|-----------|
| A (0,1) | +10 | → A' (4,1) |
| B (0,3) | +5  | → B' (2,3) |
| Edge | −1 | stays |
| Other | 0 | normal move |

### Parameters
- γ = 0.9, ε = 0.1, α = 0.2, episodes = 5000

### SARSA Update Rule
$$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s',a') - Q(s,a)]$$


In [ ]:
"""
gridworld_SARSA.py  –  SARSA (on-policy TD) for the 5×5 Gridworld MDP
Matches the textbook example: special states A=(0,1)→A'=(4,1)+10
                                              B=(0,3)→B'=(2,3)+5
γ=0.9, ε=0.1, α=0.2, episodes=5000
"""
import numpy as np
import random

# ──────────────────────────────────────────────
# Gridworld definition
# ──────────────────────────────────────────────
class Gridworld:
    ACTIONS      = ['north', 'south', 'east', 'west']
    ACTION_DELTA = {'north': (-1, 0), 'south': (1, 0),
                    'east':  (0,  1), 'west':  (0, -1)}
    ARROW        = {'north': '↑', 'south': '↓', 'east': '→', 'west': '←'}

    def __init__(self, rows=5, cols=5):
        self.rows = rows
        self.cols = cols

        # Special states (row, col)
        self.special_states = {'A': (0, 1), 'B': (0, 3)}
        self.next_states    = {'A\'': (4, 1), 'B\'': (2, 3)}
        self.special_rewards = {'A': 10, 'B': 5}

        print("Initializing Gridworld...")
        print(f"Grid size: {rows}x{cols}")
        print(f"Special_states = {{'A': {self.special_states['A']}, 'B': {self.special_states['B']}}}")
        a_prime = self.next_states["A'"]
        b_prime = self.next_states["B'"]
        print(f"Next_to_states = {{\"A'\": {a_prime}, \"B'\": {b_prime}}}")
        print(f"Special_rewards = {{'A': {self.special_rewards['A']}, 'B': {self.special_rewards['B']}}}")

    def step(self, state, action):
        r, c = state
        # Special teleport states
        if state == self.special_states['A']:
            return self.next_states["A'"], self.special_rewards['A']
        if state == self.special_states['B']:
            return self.next_states["B'"], self.special_rewards['B']

        dr, dc = self.ACTION_DELTA[action]
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.rows and 0 <= nc < self.cols:
            return (nr, nc), 0
        else:
            return state, -1   # hit wall

    def all_states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)]


# ──────────────────────────────────────────────
# SARSA
# ──────────────────────────────────────────────
def sarsa(env, gamma=0.9, epsilon=0.1, alpha=0.2, episodes=5000, steps=5000):
    # Q-table initialised to zero
    Q = {(s, a): 0.0
         for s in env.all_states()
         for a in env.ACTIONS}

    def epsilon_greedy(state):
        if random.random() < epsilon:
            return random.choice(env.ACTIONS)
        q_vals = [Q[(state, a)] for a in env.ACTIONS]
        return env.ACTIONS[int(np.argmax(q_vals))]

    print(f"\nStarting Q-learning with parameters:")
    print(f"γ = {gamma}")
    print(f" ε = {epsilon}")
    print(f" α = {alpha}")
    print(f" Episodes = {episodes}")
    print(f" Steps = {steps}")

    all_s = env.all_states()
    for ep in range(episodes):
        state  = all_s[ep % len(all_s)]
        action = epsilon_greedy(state)
        for _ in range(min(steps, 300)):
            next_state, reward = env.step(state, action)
            next_action        = epsilon_greedy(next_state)
            Q[(state, action)] += alpha * (
                reward + gamma * Q[(next_state, next_action)] - Q[(state, action)]
            )
            state  = next_state
            action = next_action

    return Q


# ──────────────────────────────────────────────
# Extract value function & policy
# ──────────────────────────────────────────────
def extract_V_and_policy(env, Q):
    V      = {}
    policy = {}
    for s in env.all_states():
        q_vals  = {a: Q[(s, a)] for a in env.ACTIONS}
        best_a  = max(q_vals, key=q_vals.get)
        V[s]    = q_vals[best_a]
        policy[s] = best_a
    return V, policy


# ──────────────────────────────────────────────
# Pretty print
# ──────────────────────────────────────────────
def print_V(env, V):
    print("\nOptimal Value Function:")
    for r in range(env.rows):
        row_str = "  ".join(f"{V[(r,c)]:5.2f}" for c in range(env.cols))
        print(row_str)

def print_policy(env, policy):
    print("\nOptimal Policy:")
    for r in range(env.rows):
        print("  " + "  ".join(f"{policy[(r,c)]:5s}" for c in range(env.cols)))

    print("\nOptimal Policy (arrows):")
    for r in range(env.rows):
        print("  " + "  ".join(env.ARROW[policy[(r,c)]] for c in range(env.cols)))


# ──────────────────────────────────────────────
# Matplotlib visualisation
# ──────────────────────────────────────────────
def visualise(env, V, policy, filename="gridworld_solution.png"):
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Value function heatmap ──
    ax = axes[0]
    grid_V = np.array([[V[(r,c)] for c in range(env.cols)] for r in range(env.rows)])
    im = ax.imshow(grid_V, cmap='RdYlGn', aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
    for r in range(env.rows):
        for c in range(env.cols):
            ax.text(c, r, f"{grid_V[r,c]:.1f}",
                    ha='center', va='center', fontsize=9, fontweight='bold')
    # mark special states
    for label, (r, c) in env.special_states.items():
        ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1,
                                    fill=False, edgecolor='blue', lw=2))
    ax.set_title("Optimal Value Function  $V_*$", fontsize=13, fontweight='bold')
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))

    # ── Policy arrows ──
    ax2 = axes[1]
    ax2.set_facecolor('#f5f5f5')
    arrow_map = {'north': (0,-0.35), 'south': (0,0.35),
                 'east':  (0.35, 0), 'west':  (-0.35, 0)}
    for r in range(env.rows):
        for c in range(env.cols):
            a  = policy[(r, c)]
            dx, dy = arrow_map[a]
            ax2.annotate("", xy=(c + dx, r + dy), xytext=(c, r),
                         arrowprops=dict(arrowstyle="->", color='#1565C0', lw=1.5))

    for label, (r, c) in env.special_states.items():
        ax2.text(c, r - 0.42, label, ha='center', va='center',
                 fontsize=9, color='darkred', fontweight='bold')

    ax2.set_xlim(-0.5, env.cols - 0.5)
    ax2.set_ylim(-0.5, env.rows - 0.5)
    ax2.set_xticks(range(env.cols))
    ax2.set_yticks(range(env.rows))
    ax2.invert_yaxis()
    ax2.grid(True, color='gray', linewidth=0.5)
    ax2.set_title("Optimal Policy  $\\pi_*$", fontsize=13, fontweight='bold')

    plt.suptitle("SARSA Gridworld Solution  (γ=0.9, ε=0.1, α=0.2)", fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {filename}")


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
if __name__ == "__main__":
    env = Gridworld()

    Q = sarsa(env, gamma=0.9, epsilon=0.1, alpha=0.2,
              episodes=5000, steps=5000)

    print("\nEvaluating optimal value function and policy...")
    V, policy = extract_V_and_policy(env, Q)

    print_V(env, V)
    print_policy(env, policy)
    visualise(env, V, policy)



In [ ]:
env = Gridworld()
Q = sarsa(env, gamma=0.9, epsilon=0.1, alpha=0.2, episodes=5000, steps=5000)
V, policy = extract_V_and_policy(env, Q)
print_V(env, V)
print_policy(env, policy)
visualise(env, V, policy)


### Comparison with Optimal (Figure 3)
The expected optimal values from the textbook are approximately:
```
22.0  24.4  22.0  19.4  17.5
19.8  22.0  19.8  17.8  16.0
17.8  19.8  17.8  16.0  14.4
16.0  17.8  16.0  14.4  13.0
14.4  16.0  14.4  13.0  11.7
```
SARSA with ε=0.1 converges close to these values. Minor differences arise because SARSA learns the **ε-soft** policy (explores 10% randomly), slightly underestimating state values vs the pure greedy optimal.
